<a href="https://colab.research.google.com/github/Desire-in-tech/Customer-Segmentation/blob/main/03_clustering_mutiple_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 3: K-Means Clustering — Multiple Features + PCA

**Dataset:** Survey of Consumer Finances (SCF) 2019  
**Method:** K-Means clustering on 5 high-variance features, visualized with PCA

In this notebook we:
- Filter households with net worth < $2M and TURNFEAR == 1
- Compute (trimmed) variance to identify the top 5 features
- Scale features with StandardScaler
- Find optimal K using inertia and silhouette curves
- Reduce dimensions to 2D with PCA for visualization

In [1]:
#Clone the GitHub repository
REPO_URL = 'https://github.com/Desire-in-tech/Customer-Segmentation.git'

!git clone {REPO_URL}

import os

os.chdir('/content/Customer-Segmentation')

#Install required packages
!pip install pandas numpy plotly scikit-learn scipy --quiet

print(f'Working directory: {os.getcwd()}')
print(f'Data file exists: {os.path.exists("data/SCF_2019.csv.gz")}')

Cloning into 'Customer-Segmentation'...
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 20 (delta 5), reused 12 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (20/20), 4.85 MiB | 13.54 MiB/s, done.
Resolving deltas: 100% (5/5), done.
Working directory: /content/Customer-Segmentation
Data file exists: True


## 1. Imports

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import mstats
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '${:,.2f}'.format)

print('Libraries loaded!')

Libraries loaded!


## 2. Wrangle Function

Filter to **TURNFEAR == 1** and **net worth < $2,000,000**.

In [3]:
def wrangle(filepath):
    """
    Load and wrangle SCF 2019 data.

    Filters:
    - Implicate 1 only (every 5th row)
    - TURNFEAR == 1 (turned down or feared credit denial)
    - NETWORTH < 2,000,000 (focus on lower/middle wealth households)

    Parameters
    ----------
    filepath : str
        Path to compressed CSV (SCF_2019.csv.gz)

    Returns
    -------
    pd.DataFrame
    """
    df = pd.read_csv(filepath, compression='gzip', index_col=0)

    # Keep implicate 1
    df = df.iloc[::5].reset_index(drop=True)

    # Filter: TURNFEAR == 1
    df = df[df['TURNFEAR'] == 1].copy()

    # Filter: net worth < $2M
    df = df[df['NETWORTH'] < 2_000_000].copy()

    return df.reset_index(drop=True)


df = wrangle('data/SCF_2019.csv.gz')

print(f'Shape: {df.shape}')
print(f'Net worth range: ${df["NETWORTH"].min():,.0f} to ${df["NETWORTH"].max():,.0f}')
df.head()

Shape: (882, 357)
Net worth range: $-1,029,421 to $1,836,413


,YY1,Y1,WGT,HHSEX,AGE,AGECL,EDUC,EDCL,MARRIED,KIDS,...,NWCAT,INCCAT,ASSETCAT,NINCCAT,NINC2CAT,NWPCTLECAT,INCPCTLECAT,NINCPCTLECAT,INCQRTCAT,NINCQRTCAT
0,2,21,"$3,790.48",1,50,3,8,2,1,3,...,1,2,1,2,1,1,4,4,2,2
1,23,231,"$5,424.99",1,69,5,11,3,2,0,...,1,1,1,2,1,3,2,3,1,1
2,31,311,"$4,730.10",1,31,1,9,3,1,2,...,1,3,2,3,2,1,6,6,3,3
3,37,371,"$4,705.03",1,45,3,2,1,1,0,...,1,2,1,2,1,3,4,4,2,2
4,40,401,"$5,620.47",1,68,5,12,4,2,0,...,2,2,3,2,1,5,4,3,2,2


## 3. Identify High-Variance Features

In [4]:
# Select only numeric columns (exclude categorical/flag variables)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Exclude obvious non-feature columns
exclude = ['YY1', 'Y1', 'WGT', 'TURNFEAR', 'RACECL4', 'RACECL', 'HHSEX',
           'MARRIED', 'KIDS', 'OCCAT1', 'OCCAT2', 'EDUC', 'INCOME_RANK']
feature_cols = [c for c in numeric_cols if c not in exclude]

# Compute variance for all numeric features
variance_series = df[feature_cols].var().sort_values(ascending=False)

# Top 10 features by variance
top_ten_var = variance_series.head(10)

print('Top 10 features by variance:')
print(top_ten_var.to_string())

Top 10 features by variance:
ASSET      $96,535,333,971.90
NFIN       $72,154,772,565.59
NETWORTH   $53,066,491,493.26
NHNFIN     $37,346,941,397.68
HOUSES     $30,028,715,125.72
DEBT       $24,667,851,911.56
KGTOTAL    $16,399,974,370.28
BUS        $15,156,861,390.27
ACTBUS     $15,074,179,599.76
PLOAN1     $15,048,303,135.69


## 4. Bar Chart: High-Variance Features

In [5]:
fig = px.bar(
    x=top_ten_var.values,
    y=top_ten_var.index,
    orientation='h',
    title='SCF: High Variance Features',
    labels={'x': 'Variance', 'y': 'Feature'},
    color=top_ten_var.values,
    color_continuous_scale='Blues'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

## 5. Boxplot: Non-Home, Non-Financial Assets (NHNFIN)

In [6]:
fig = px.box(
    df,
    x='NHNFIN',
    title='Distribution of Non-home, Non-Financial Assets',
    labels={'NHNFIN': 'Value [$]'},
    color_discrete_sequence=['#636EFA']
)
fig.show()

print(f'NHNFIN - Skewness: {df["NHNFIN"].skew():.2f}')
print('(values > 1.0 indicate right-skewed distribution — extreme high values)')

NHNFIN - Skewness: 7.60
(values > 1.0 indicate right-skewed distribution — extreme high values)


## 6. Trimmed Variance

Since the raw data is highly skewed, we use **trimmed variance** (removing extreme values before computing variance) to identify features that truly vary across typical households.

In [8]:
def trimmed_variance(x, proportiontocut=0.1):
    """
    Compute variance after trimming the top and bottom proportiontocut
    of values from a series.

    Parameters
    ----------
    x : array-like
    proportiontocut : float (default 0.1 = 10% from each tail)

    Returns
    -------
    float : trimmed variance
    """
    x_trimmed = mstats.trimr(x, (proportiontocut, proportiontocut))
    return np.var(x_trimmed.compressed())


# Compute trimmed variance for all features
trim_var_values = {
    col: trimmed_variance(df[col].dropna().values)
    for col in feature_cols
    if df[col].notna().sum() > 100  # need enough data
}
trim_var_series = pd.Series(trim_var_values).sort_values(ascending=False)

# Top 10 by trimmed variance
top_ten_trim_var = trim_var_series.head(10)

print('Top 10 features by TRIMMED variance:')
print(top_ten_trim_var.to_string())

Top 10 features by TRIMMED variance:
ASSET      $15,222,871,317.11
NFIN       $10,908,171,924.63
HOUSES      $6,464,032,790.19
DEBT        $4,129,276,671.34
NETWORTH    $4,094,597,508.90
PLOAN1      $1,921,371,774.42
MRTHEL      $1,843,492,468.90
NH_MORT     $1,780,090,061.57
HOMEEQ        $944,320,624.06
WAGEINC       $745,827,064.24


## 7. Bar Chart: Trimmed High-Variance Features

In [9]:
fig = px.bar(
    x=top_ten_trim_var.values,
    y=top_ten_trim_var.index,
    orientation='h',
    title='SCF: High Trimmed-Variance Features',
    labels={'x': 'Trimmed Variance', 'y': 'Feature'},
    color=top_ten_trim_var.values,
    color_continuous_scale='Oranges'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

## 8. Select Top 5 High-Variance Columns

In [10]:
# Top 5 features with highest trimmed variance
high_var_cols = top_ten_trim_var.head(5).index.tolist()

print('Top 5 high-variance features selected for clustering:')
for i, col in enumerate(high_var_cols, 1):
    print(f'  {i}. {col}')

Top 5 high-variance features selected for clustering:
  1. ASSET
  2. NFIN
  3. HOUSES
  4. DEBT
  5. NETWORTH


## 9. Feature Matrix and Summary Statistics

In [11]:
# Create feature matrix
X = df[high_var_cols].copy()

print(f'Feature matrix shape: {X.shape}')

# Summary statistics before scaling
X_summary = pd.DataFrame({
    'Mean': X.mean(),
    'Std Dev': X.std()
})

print('\nX Summary (before scaling):')
print(X_summary.to_string())

Feature matrix shape: (882, 5)

X Summary (before scaling):
                Mean     Std Dev
ASSET    $165,975.24 $310,701.36
NFIN     $132,348.22 $268,616.40
HOUSES    $84,062.76 $173,287.95
DEBT      $83,555.86 $157,060.03
NETWORTH  $82,419.39 $230,361.65


## 10. StandardScaler: Normalize Features

In [12]:
# Create and fit StandardScaler
scaler = StandardScaler()
X_scaled_array = scaler.fit_transform(X)

# Put into DataFrame
X_scaled = pd.DataFrame(X_scaled_array, columns=high_var_cols)

# Summary after scaling
X_scaled_summary = pd.DataFrame({
    'Mean': X_scaled.mean(),
    'Std Dev': X_scaled.std()
})

print('X_scaled Summary (after StandardScaler):')
print(X_scaled_summary.round(6).to_string())
print('\n(Mean ≈ 0, Std Dev ≈ 1 after scaling)')

X_scaled Summary (after StandardScaler):
           Mean  Std Dev
ASSET    $-0.00    $1.00
NFIN      $0.00    $1.00
HOUSES    $0.00    $1.00
DEBT     $-0.00    $1.00
NETWORTH $-0.00    $1.00

(Mean ≈ 0, Std Dev ≈ 1 after scaling)


## 11. K-Means Pipeline: Evaluate K from 2 to 12

In [13]:
n_clusters_range = range(2, 13)
inertia_errors = []
silhouette_scores = []

for k in n_clusters_range:
    # Pipeline: StandardScaler + KMeans
    pipeline = make_pipeline(
        StandardScaler(),
        KMeans(n_clusters=k, random_state=42, n_init=10)
    )
    pipeline.fit(X)

    # Inertia from the KMeans step
    inertia = pipeline.named_steps['kmeans'].inertia_
    inertia_errors.append(inertia)

    # Silhouette score (on scaled data)
    labels_k = pipeline.named_steps['kmeans'].labels_
    X_scaled_temp = pipeline.named_steps['standardscaler'].transform(X)
    sil = silhouette_score(X_scaled_temp, labels_k)
    silhouette_scores.append(sil)

    print(f'K={k:2d} | Inertia: {inertia:>15,.2f} | Silhouette: {sil:.4f}')

print('\nEvaluation complete!')

K= 2 | Inertia:        2,174.75 | Silhouette: 0.7249
K= 3 | Inertia:        1,471.89 | Silhouette: 0.6956
K= 4 | Inertia:        1,186.28 | Silhouette: 0.6861
K= 5 | Inertia:        1,008.69 | Silhouette: 0.6432
K= 6 | Inertia:          871.14 | Silhouette: 0.6454
K= 7 | Inertia:          753.69 | Silhouette: 0.6435
K= 8 | Inertia:          640.94 | Silhouette: 0.6438
K= 9 | Inertia:          589.23 | Silhouette: 0.6427
K=10 | Inertia:          539.07 | Silhouette: 0.5705
K=11 | Inertia:          487.88 | Silhouette: 0.6021
K=12 | Inertia:          446.93 | Silhouette: 0.5846

Evaluation complete!


## 12. Inertia vs. Number of Clusters

In [14]:
fig = px.line(
    x=list(n_clusters_range),
    y=inertia_errors,
    markers=True,
    title='K-Means Model: Inertia vs Number of Clusters',
    labels={'x': 'Number of Clusters', 'y': 'Inertia'}
)
fig.update_traces(line_color='#636EFA', marker_color='#EF553B', marker_size=9)
fig.update_layout(xaxis=dict(tickmode='linear', tick0=2, dtick=1))
fig.show()

## 13. Silhouette Score vs. Number of Clusters

In [15]:
fig = px.line(
    x=list(n_clusters_range),
    y=silhouette_scores,
    markers=True,
    title='K-Means Model: Silhouette Score vs Number of Clusters',
    labels={'x': 'Number of Clusters', 'y': 'Silhouette Score'}
)
fig.update_traces(line_color='#00CC96', marker_color='#EF553B', marker_size=9)
fig.update_layout(xaxis=dict(tickmode='linear', tick0=2, dtick=1))
fig.show()

best_k_idx = np.argmax(silhouette_scores)
best_k = list(n_clusters_range)[best_k_idx]
print(f'Best K by silhouette: K={best_k} (score={silhouette_scores[best_k_idx]:.4f})')

Best K by silhouette: K=2 (score=0.7249)


## 14. Final Model

In [16]:
# Build final model — adjust BEST_K based on your plots
BEST_K = 2

final_model = make_pipeline(
    StandardScaler(),
    KMeans(n_clusters=BEST_K, random_state=42, n_init=10)
)
final_model.fit(X)

# Extract labels
labels = final_model.named_steps['kmeans'].labels_
final_inertia = final_model.named_steps['kmeans'].inertia_
X_scaled_final = final_model.named_steps['standardscaler'].transform(X)
final_sil = silhouette_score(X_scaled_final, labels)

print(f'Final Model (K={BEST_K})')
print(f'  Inertia:          {final_inertia:,.2f}')
print(f'  Silhouette Score: {final_sil:.4f}')

# Cluster sizes
for c, cnt in pd.Series(labels).value_counts().sort_index().items():
    print(f'  Cluster {c}: {cnt:,} ({cnt/len(labels)*100:.1f}%)')

Final Model (K=2)
  Inertia:          2,174.75
  Silhouette Score: 0.7249
  Cluster 0: 758 (85.9%)
  Cluster 1: 124 (14.1%)


## 15. Mean Household Finances by Cluster

In [17]:
# DataFrame of mean values per cluster
xgb = X.copy()
xgb['Cluster'] = labels
xgb = xgb.groupby('Cluster')[high_var_cols].mean().reset_index()
xgb['Cluster'] = xgb['Cluster'].astype(str)

print('Mean feature values per cluster:')
print(xgb.to_string(index=False))

# Melt for grouped bar chart
xgb_melted = xgb.melt(id_vars='Cluster', var_name='Feature', value_name='Mean Value ($)')

fig = px.bar(
    xgb_melted,
    x='Cluster',
    y='Mean Value ($)',
    color='Feature',
    barmode='group',
    title='Mean Household Finances by Cluster',
    labels={'Cluster': 'Cluster', 'Mean Value ($)': 'Value [$]'},
    color_discrete_sequence=px.colors.qualitative.Plotly
)
fig.show()

Mean feature values per cluster:
Cluster       ASSET        NFIN      HOUSES        DEBT    NETWORTH
      0  $67,398.50  $50,977.41  $31,017.92  $45,193.55  $22,204.95
      1 $768,565.37 $629,760.07 $408,320.72 $318,060.94 $450,504.43


## 16. PCA: Reduce to 2 Dimensions

In [18]:
# Create PCA transformer and reduce to 2 components
pca = PCA(n_components=2, random_state=42)
X_pca_array = pca.fit_transform(X_scaled_final)

X_pca = pd.DataFrame(X_pca_array, columns=['PC1', 'PC2'])

print(f'PCA explained variance ratio:')
print(f'  PC1: {pca.explained_variance_ratio_[0]*100:.1f}%')
print(f'  PC2: {pca.explained_variance_ratio_[1]*100:.1f}%')
print(f'  Total: {sum(pca.explained_variance_ratio_)*100:.1f}%')

X_pca.head()

PCA explained variance ratio:
  PC1: 77.9%
  PC2: 16.3%
  Total: 94.1%


,PC1,PC2
0,$-1.04,$0.05
1,$-1.03,$0.16
2,$-0.69,$-0.16
3,$-1.02,$0.17
4,$0.37,$-0.18


## 17. PCA Scatter Plot of Clusters

In [19]:
X_pca['labels'] = labels.astype(str)

fig = px.scatter(
    X_pca,
    x='PC1',
    y='PC2',
    color='labels',
    title='PCA Representation of Clusters',
    labels={'PC1': 'PC1', 'PC2': 'PC2', 'labels': 'Cluster'},
    opacity=0.6,
    color_discrete_sequence=px.colors.qualitative.Plotly
)
fig.update_layout(
    annotations=[
        dict(
            text=f'Total variance explained: {sum(pca.explained_variance_ratio_)*100:.1f}%',
            xref='paper', yref='paper',
            x=0.01, y=0.99,
            showarrow=False,
            bgcolor='white',
            bordercolor='gray',
            borderwidth=1
        )
    ]
)
fig.show()

## 18. Cluster Profile Summary

In [20]:
# Add cluster labels to original df for profiling
df_clustered = df.copy()
df_clustered['Cluster'] = labels

profile_cols = ['INCOME', 'NETWORTH', 'ASSET', 'DEBT', 'AGE'] + high_var_cols
profile_cols = list(dict.fromkeys(profile_cols))  # deduplicate

profile = df_clustered.groupby('Cluster')[profile_cols].agg(['mean', 'median'])
profile.columns = ['_'.join(col) for col in profile.columns]

print('Cluster Profiles (mean | median):')
print(profile.T.to_string())

Cluster Profiles (mean | median):
Cluster                  0           1
INCOME_mean     $50,497.62 $120,151.10
INCOME_median   $41,308.66 $101,501.28
NETWORTH_mean   $22,204.95 $450,504.43
NETWORTH_median  $6,091.62 $343,049.42
ASSET_mean      $67,398.50 $768,565.37
ASSET_median    $20,251.32 $595,773.56
DEBT_mean       $45,193.55 $318,060.94
DEBT_median     $12,832.40 $266,553.46
AGE_mean            $43.26      $50.98
AGE_median          $41.00      $50.00
NFIN_mean       $50,977.41 $629,760.07
NFIN_median     $12,751.26 $489,416.48
HOUSES_mean     $31,017.92 $408,320.72
HOUSES_median        $0.00 $368,627.28
